In [ ]:
"""Trabalho_Prático_—_Comparação_de_Algoritmos_de_Aprendizado_de_Máquina_Supervisionado_Grupo7_Final.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1dumkx3Klosb7dTJoxeYF_HKUsbxi4yDZ

# Trabalho Prático — Comparação de Algoritmos de Aprendizado de Máquina Supervisionado (Grupo 7)

##Objetivo

Aplicar os conceitos de aprendizado supervisionado abordados em aula: i) KNN, ii) SVM, iii) Árvores de Decisão e iv) MLP na resolução de um problema real de **Classificação** ou **Regressão**. O trabalho tem como foco a análise comparativa de desempenho entre algoritmos, bem como a discussão crítica dos resultados obtidos.

⚠️

O notebook entregue deve ser autocontido e totalmente executável. Isso significa que:

* Todas as bibliotecas necessárias devem ser importadas explicitamente no início do notebook.

* Exceto pelo carregamento (upload) do dataset (i.e data.zip), o restante código deve ser executável do início ao fim sem intervenções manuais, sem necessidade de ajustes, inserção de inputs, descompactações e/ou modificações de caminho de arquivos.

* Todos os arquivos necessários (por exemplo, dataset processado) devem estar acessíveis via link público para download.

* O notebook deve gerar todas as saídas esperadas automaticamente, incluindo métricas, gráficos e relatórios, sem dependência de ações externas.


⚠️ Exceto pelo carregamento do dataset (i.e data.zip), trabalhos que exigirem intervenção manual (como importação adicional de bibliotecas, inserção de inputs ou correções de código) serão desconsiderados na avaliação.

##Identificação da Equipe:

* **ID GRUPO CANVAS**: 2026.2BI 7
* **Membro 1**: Andre da Silva
* **Membro 2**: Bruno Fernandes do Nascimento
* **Membro 3**: Edgar de Oliveira Cabral Filho
* **Membro 4**: Lucas de Almeida Ferreira
* **Membro 5**: Nicolas Livrari


⚠️ Além da identificação, **todos os membros do grupo** devem estar devidamente alocados no ambiente virtual (AVA), em:
**Pessoas → Grupos → Trabalho Final.**

⚠️ Apenas **uma submissão por grupo** é necessária. Entretanto, é imprescindível que **todos os integrantes estejam alocados** no respectivo grupo para que a atribuição do envio seja registrada corretamente.

##Seed
"""

In [ ]:
import os
import numpy as np
import random
import sklearn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import RBFSampler
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
# Definição de semente para reprodutibilidade
SEED = 42

In [ ]:
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
# Configurações iniciais
pd.set_option("display.max_columns", None)

In [ ]:
"""#Definição e Descrição do Dataset

### Seleção do Dataset

Selecione um **dataset público** disponível no [Kaggle – Categoria Classification](https://www.kaggle.com/datasets?tags=13302-Classification), que contenha **rótulos (labels)**, caracterizando uma tarefa de **classificação binária** ou **multiclasse**.

O dataset escolhido deve apresentar **quantidade suficiente de instâncias e atributos** para permitir uma análise estatística e experimental significativa.

Caso o dataset tenha sido **pré-processado** (por exemplo: limpeza, seleção de atributos, codificação ou balanceamento), o **arquivo resultante** deve ser compartilhado em um link público (Google Drive, Dropbox, etc.), de modo que possa ser baixado e executado sem restrições.

> 🔗 **LINK PARA O DATASET:**
> *(https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset?select=diabetes_binary_health_indicators_BRFSS2015.csv)* <<==

⚠️ **Observação:** Trabalhos sem o link de acesso ao dataset não serão avaliados, pois impedem a reprodutibilidade do experimento.



---

##Descreva sucitamente o dataset (aplicação, desafios, etc)

Este dataset contém dados de entrevistados norte americanos que apresentam tendência a desenvolver diabetes. É conduzido pelo sistema americano The Behavioral Risk Factor Surveillance System (BRFSS) e são coletados anualmente desde 1984 a fim de promover ações de prevenção e tratamento da doença. O dataset original contém mais de 330 variáveis, porém, apenas 21 delas estão disponíveis publicamente.

Estes dados servirão como base para classificar potenciais portadores de diabetes através da aplicação de ferramentas de Machine Learning. O principal desafio está relacionado a reproduzir um resultado condizente utilizando as variáveis (features) disponíveis.

###Dicionário de dados:

| # | Variável Original | Nome Traduzido | Tipo | Descrição | Valores Possíveis |
|---|-------------------|----------------|------|-----------|-------------------|
| 1 | `Diabetes_binary` | `Target` | Binária | Indicador de diabetes (variável alvo) | 0 = Não diabético<br>1 = Pré-diabético ou diabético |
| 2 | `HighBP` | `PressaoAlta` | Binária | Diagnóstico de pressão alta | 0 = Não<br>1 = Sim |
| 3 | `HighChol` | `ColesterolAlto` | Binária | Diagnóstico de colesterol alto | 0 = Não<br>1 = Sim |
| 4 | `CholCheck` | `VerificouColesterol` | Binária | Verificou o colesterol nos últimos 5 anos | 0 = Não<br>1 = Sim |
| 5 | `BMI` | `IMC` | Numérica | Índice de Massa Corporal | Valor contínuo (ex: 18.0 a 98.0) |
| 6 | `Smoker` | `Fumante` | Binária | Já fumou pelo menos 100 cigarros na vida | 0 = Não<br>1 = Sim |
| 7 | `Stroke` | `Derrame` | Binária | Já teve derrame (AVC) | 0 = Não<br>1 = Sim |
| 8 | `HeartDiseaseorAttack` | `DoencaCardiaca` | Binária | Doença coronariana ou infarto do miocárdio | 0 = Não<br>1 = Sim |
| 9 | `PhysActivity` | `AtividadeFisica` | Binária | Praticou atividade física nos últimos 30 dias | 0 = Não<br>1 = Sim |
| 10 | `Fruits` | `ConsomeFrutas` | Binária | Consome frutas pelo menos 1 vez ao dia | 0 = Não<br>1 = Sim |
| 11 | `Veggies` | `ConsomeVegetais` | Binária | Consome vegetais pelo menos 1 vez ao dia | 0 = Não<br>1 = Sim |
| 12 | `HvyAlcoholConsump` | `ConsomeAlcoolPesado` | Binária | Consumo pesado de álcool (homens: >14 doses/sem; mulheres: >7 doses/sem) | 0 = Não<br>1 = Sim |
| 13 | `AnyHealthcare` | `PlanoSaude` | Binária | Possui algum tipo de plano ou cobertura de saúde | 0 = Não<br>1 = Sim |
| 14 | `NoDocbcCost` | `NaoConsultMedicoCusto` | Binária | Deixou de consultar médico nos últimos 12 meses por causa do custo | 0 = Não<br>1 = Sim |
| 15 | `GenHlth` | `AutoAvaliacaoSaude` | Ordinal | Autoavaliação do estado geral de saúde | 1 = Excelente<br>2 = Muito boa<br>3 = Boa<br>4 = Regular<br>5 = Ruim |
| 16 | `MentHlth` | `DiasSaudeMentalRuim` | Discreta | Nº de dias (0-30) com saúde mental não boa no último mês | 0 a 30 |
| 17 | `PhysHlth` | `DiasSaudeFisicaRuim` | Discreta | Nº de dias (0-30) com saúde física não boa no último mês | 0 a 30 |
| 18 | `DiffWalk` | `DificuldadeCaminhar` | Binária | Dificuldade séria para caminhar ou subir escadas | 0 = Não<br>1 = Sim |
| 19 | `Sex` | `Genero` | Binária | Sexo biológico do respondente | 0 = Feminino<br>1 = Masculino |
| 20 | `Age` | `Idade` | Ordinal | Faixa etária do respondente | 1 = 18-24<br>2 = 25-29<br>3 = 30-34<br>4 = 35-39<br>5 = 40-44<br>6 = 45-49<br>7 = 50-54<br>8 = 55-59<br>9 = 60-64<br>10 = 65-69<br>11 = 70-74<br>12 = 75-79<br>13 = 80+ |
| 21 | `Education` | `Escolaridade` | Ordinal | Nível de escolaridade | 1 = Nunca frequentou escola<br>2 = Fundamental incompleto<br>3 = Fundamental completo<br>4 = Ensino médio incompleto<br>5 = Ensino médio completo<br>6 = Superior (graduação+) |
| 22 | `Income` | `Renda` | Ordinal | Faixa de renda familiar anual | 1 = Menos de US 10.000<br>2 = US 10.000-14.999<br>3 = US 15.000-19.999<br>4 = US 20.000-24.999<br>5 = US 25.000-34.999<br>6 = US 35.000-49.999<br>7 = US 50.000-74.999<br>8 = US 75.000+ |

**Total de variáveis:** 22 (1 target + 21 features)  
**Total de registros:** 253.680


---

#Preparação dos Dados

Nesta seção, desenvolva **rotinas** para o tratamento do dataset selecionado, contemplando as seguintes etapas:

* **Descompactação e Importação:** implemente a leitura direta do arquivo (CSV, XLSX, ZIP, etc.), garantindo o correto carregamento dos dados.
* **Limpeza e Pré-processamento:** realize as etapas necessárias de tratamento, como remoção de valores ausentes, codificação de variáveis categóricas, normalização ou padronização, e balanceamento das classes, quando aplicável.
* **Divisão dos Conjuntos:** separe o dataset em **treino, validação e teste**, utilizando o método **Holdout** de forma adequada à natureza do problema.
* **Análise Exploratória:** elabore **gráficos e estatísticas descritivas** que permitam compreender a distribuição das variáveis, a correlação entre atributos e o equilíbrio entre as classes.
"""

In [ ]:
# Instalando gdown para download automático do dataset via Google Drive
!pip install gdown -q

In [ ]:
import gdown

In [ ]:
# ID do arquivo no Google Drive (acesso público, leitor)
FILE_ID = "1z4I2kWQz9G-NdwJ6ryNJvVYhjCbqZtux"
DADOS = "diabetes_binary_health_indicators_BRFSS2015.csv"

In [ ]:
# Faz o download apenas se o arquivo ainda não existir no ambiente
import os
if not os.path.exists(DADOS):
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", DADOS, quiet=False)

In [ ]:
# Carrega o dataset
df = pd.read_csv(DADOS)

In [ ]:
df.head(1)

Traduzindo variáveis para português

In [ ]:
df.rename(columns={
  "Diabetes_binary": "Target",
  "HighBP": "PressaoAlta",
  "HighChol": "ColesterolAlto",
  "CholCheck": "VerificouColesterol",
  "BMI": "IMC",
  "Smoker": "Fumante",
  "Stroke": "Derrame",
  "HeartDiseaseorAttack": "DoencaCardiaca",
  "PhysActivity": "AtividadeFisica",
  "Fruits": "ConsomeFrutas",
  "Veggies": "ConsomeVegetais",
  "HvyAlcoholConsump": "ConsomeAlcoolPesado",
  "AnyHealthcare": "PlanoSaude",
  "NoDocbcCost": "NaoConsultMedicoCusto",
  "GenHlth": "AutoAvaliacaoSaude",
  "MentHlth": "DiasSaudeMentalRuim",
  "PhysHlth": "DiasSaudeFisicaRuim",
  "DiffWalk": "DificuldadeCaminhar",
  "Sex": "Genero",
  "Age": "Idade",
  "Education": "Escolaridade",
  "Income": "Renda"
}, inplace=True)

In [ ]:
df.head()

In [ ]:
# Colunas e valores nulos
df.info()

In [ ]:
print("Este dataset contém {} linhas e {} colunas.".format(*df.shape))

Separando variáveis em categóricas e numéricas

In [ ]:
# Variáveis numéricas (quantitativas discretas e contínuas)
var_numericas = [
  "IMC",
  "DiasSaudeMentalRuim",
  "DiasSaudeFisicaRuim"
]

In [ ]:
# Target
var_target = ["Target"]

In [ ]:
# Variáveis categóricas (qualitativas nominais e ordinais)
var_ordinais = [
  "AutoAvaliacaoSaude",
  "Idade",
  "Escolaridade",
  "Renda"
]

In [ ]:
var_nominais = [var for var in df.columns if var not in (var_ordinais + var_numericas + var_target)]

In [ ]:
# Início da análise exploratória
df["Target"].value_counts(normalize=True).mul(100)

In [ ]:
"""> O dataset é desbalanceado, 86% de casos negativos para diabetes vs ~14% de casos positivos"""

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6), ncols=1, nrows=2, gridspec_kw={
    "height_ratios": (0.2, 0.8),
    "hspace": 0.02
}, sharex=True)

In [ ]:
b = sns.boxplot(data=df, x="IMC", ax=ax[0], hue="Target")
b.tick_params(axis="both", length=0)
b.legend().remove()

In [ ]:
h = sns.histplot(data=df, x="IMC", ax=ax[1], hue="Target", kde=True, bins="sturges", multiple="stack", stat="density")
h.xaxis.set_major_locator(mticker.MultipleLocator(base=5))
h.set_ylabel("Densidade")
h.tick_params(axis="both", length=0)
h.annotate("IMC:\n-> abaixo de 18,5 é considerado baixo peso\n-> entre 18,5 e 24,9 é peso normal\n-> 25 a 29,9 é sobrepeso\n-> 30 ou mais indica obesidade", xy=(42, 0.06))
h.annotate("É possível observar que a maior parte de pacientes diabéticos (target=1) estão entre 25 e 40\nisto é, entre sobrepeso e obesidade.", xy=(42, 0.05))
h.axvspan(25, 40, alpha=0.25, color="C3")

In [ ]:
fig.suptitle("Visualização IMC x Target")

In [ ]:
plt.show()

In [ ]:
"""> IMC abaixo de 18,5 é considerado baixo peso; entre 18,5 e 24,9 é peso normal; 25 a 29,9 é sobrepeso; e 30 ou mais indica obesidade. É possível observar que a maior parte dos pacientes diabéticos (target=1) estão entre 25 e 40, isto é, entre **sobrepeso** e **obesidade**."""

In [ ]:
# Dias de saúde física ou mental ruim nos últimos 30 dias
df.groupby("Target")[["DiasSaudeFisicaRuim", "DiasSaudeMentalRuim"]].describe()\
.T.rename(columns={0.0: "Não diabético", 1.0: "Diabético"})

In [ ]:
"""> Pode-se observar que pelo menos 75% dos pacientes diabéticos tiveram ao menos duas semanas de saúde física ruim no último mês. Além disso, ao menos 3 dias de saúde mental prejudicada."""

In [ ]:
# Criando dataframe para plotar as variáveis nominais
df_plot_nominais = df[var_nominais + var_target].replace({0: "Não", 1: "Sim"})

In [ ]:
df_plot_nominais.replace(
    {"Genero": {"Sim": "Masculino", "Não": "Feminino"},
     "Target": {"Sim": "Diabético", "Não": "Não diabético"}},
    inplace=True
    )

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10), nrows=4, ncols=4, gridspec_kw={"hspace": 0.35})

In [ ]:
for i, var in enumerate(var_nominais):
  b = sns.countplot(data=df_plot_nominais, x=var, ax=ax.flatten()[i], hue="Target")
  b.tick_params(axis="both", length=0)
  b.set_yticks([])
  b.set_ylabel("")
  b.set_xlabel("")
  b.set_title(var, fontsize=10)
  b.set_frame_on(False)

  for container in b.containers:
    b.bar_label(container, fontsize=9, label_type="edge")

  legend = b.get_legend()
  legend.remove()

In [ ]:
textos_legenda = [texts.get_text() for texts in legend.get_texts()]

In [ ]:
fig.legend(textos_legenda, handles=legend.legend_handles, bbox_to_anchor=(0.6, 0.96), title="Target", ncols=2)

In [ ]:
fig.suptitle("Total de pacientes diabéticos e não diabéticos por condição por classe")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10), nrows=4, ncols=4, gridspec_kw={"hspace": 0.35})

In [ ]:
for i, var in enumerate(var_nominais):
  b = sns.countplot(data=df_plot_nominais.query("Target == 'Diabético'"), x=var, ax=ax.flatten()[i], hue="Target")
  b.tick_params(axis="both", length=0)
  b.set_yticks([])
  b.set_ylabel("")
  b.set_xlabel("")
  b.set_title(var, fontsize=10)
  b.set_frame_on(False)

  for container in b.containers:
    b.bar_label(container, fontsize=9, label_type="edge")

  legend = b.get_legend()
  legend.remove()

In [ ]:
textos_legenda = [texts.get_text() for texts in legend.get_texts()]

In [ ]:
fig.legend(textos_legenda, handles=legend.legend_handles, bbox_to_anchor=(0.55, 0.96), title="Target", ncols=2)

In [ ]:
fig.suptitle("Total de pacientes diabéticos por condição")
plt.show()

In [ ]:
"""> Pode-se observar que a maior parte dos pacientes diabéticos são mulheres e sofrem de pressão alta, colesterol alto e são fumantes. Por outro lado, observa-se que uma grande fração faz exames regulares, possuem plano de saúde, praticam atividade física regularmente e consomem alimentos saudáveis. Isto demonstra busca por uma vida mais saudável, mesmo sendo portador da doença.

---

#Implementação dos Modelos

Treinar e testar os seguintes algoritmos:

* K-Nearest Neighbors (KNN)

* Support Vector Machine (SVM)

* Decision Tree (Árvore de Decisão)

* Multi-Layer Perceptron (MLP)

### Holdout
"""

In [ ]:
X = df.iloc[:,1:]
y = df["Target"]

In [ ]:
## Separando o conjunto de teste (20% dos dados)
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [ ]:
# Dividindo o restante entre treino e validação (pegando 12.5% do total para validação)
# 12.5% de 80% é aproximadamente 15% do dataset inicial
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, test_size=0.125, random_state=SEED)

In [ ]:
print(f"Tamanho do treino: {len(X_train)}")
print(f"Tamanho da validação: {len(X_val)}")
print(f"Tamanho do teste: {len(X_test)}")

In [ ]:
"""### Pré processamento"""

SimpleImputer --> usado para preencher valores faltantes
OneHotEncoder --> usado para variáveis nominais
StandardScaler --> usado para variáveis numéricas

In [ ]:
preprocessing = ColumnTransformer([
    ("var_nominais", ImbPipeline([
        ("imputer", SimpleImputer(strategy="constant")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ]), var_nominais),

    ("var_numericas", ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), var_numericas),

    ("var_ordinais", SimpleImputer(strategy="median"), var_ordinais)
])

In [ ]:
"""### Algoritmos"""

In [ ]:
# Armazenar resultados
models_result = []

In [ ]:
# KNN
knn_pipe = ImbPipeline([
   ("preprocessing", preprocessing),
   ("smote", SMOTE(random_state=SEED)), # Utilizando SMOTE
   ("classifier", KNeighborsClassifier(
       n_neighbors=5,
       weights='distance',
       metric='euclidean',
       n_jobs=-1
   ))
])

In [ ]:
# Treinando o modelo
knn_pipe.fit(X_train, y_train)

In [ ]:
# Obter probabilidades
y_val_prob = knn_pipe.predict_proba(X_val)[:, 1]

In [ ]:
# Prevendo valor de y
threshold = 0.65
y_val_pred_knn = (y_val_prob > threshold).astype(int)

In [ ]:
# Salvando
models_result.append({
    "model": "KNN",
    "y_pred": y_val_pred_knn
})

In [ ]:
# SVM
svm_pipe = ImbPipeline([
   ("preprocessing", preprocessing),
   ("smote", SMOTE(random_state=SEED, k_neighbors=3, sampling_strategy=0.5)), # Utilizando SMOTE
   ("classifier", CalibratedClassifierCV(
        LinearSVC(C=1.0, random_state=42, class_weight='balanced', max_iter=1000),
        cv=3
   ))
])

In [ ]:
# Treinando o modelo
svm_pipe.fit(X_train, y_train)

In [ ]:
# Prevendo valor de y
y_val_pred_svm = svm_pipe.predict(X_val)

In [ ]:
# Salvando
models_result.append({
    "model": "SVM",
    "y_pred": y_val_pred_svm
})

In [ ]:
# Decision Tree
dt_pipe = ImbPipeline([
   ("preprocessing", preprocessing),
   ("smote", SMOTE(random_state=SEED)), # Utilizando SMOTE
   ("classifier", DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=20,
        random_state=SEED
    ))
])

In [ ]:
# Treinando o modelo
dt_pipe.fit(X_train, y_train)

In [ ]:
# Prevendo valor de y
y_val_pred_dt = dt_pipe.predict(X_val)

In [ ]:
# Salvando
models_result.append({
    "model": "DT",
    "y_pred": y_val_pred_dt
})

In [ ]:
# MLP
mlp_pipe = ImbPipeline([
   ("preprocessing", preprocessing),
   ("smote", SMOTE(random_state=SEED)), # Utilizando SMOTE
   ("classifier", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=300,
        early_stopping=True,
        random_state=SEED
    ))
])

In [ ]:
# Treinando o modelo
mlp_pipe.fit(X_train, y_train)

In [ ]:
# Prevendo valor de y
y_val_pred_mlp = mlp_pipe.predict(X_val)

In [ ]:
# Salvando
models_result.append({
    "model": "MLP",
    "y_pred": y_val_pred_mlp
})

In [ ]:
"""---

#Avaliação dos modelos


* Calcular e comparar as métricas: Acurácia, Precisão, Recall e F1-Score.


* Plotar e comentar as matrizes de confusão.


* Discutir o impacto dos hiperparâmetros e preprocessamentos aplicados.

Aqui você pode elaborar gráficos, analises de resultados da maneira que desejar. No entanto, espera-se uma breve discussão dos resultados e análises críticas.
"""

=== Função auxiliar para avaliação visual dos modelos ===
Gera: cards com métricas principais (Acurácia, Precisão, Recall, F1)
e matriz de confusão em formato de heatmap, além do classification_report completo

In [ ]:
def avaliar_modelo(nome, y_true, y_pred, y_prob=None):
    # Calcula as métricas principais
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": (1, 1.2)})

    # --- Painel esquerdo: cards com as métricas ---
    metricas = {"Acurácia": acc, "Precisão": prec, "Recall": rec, "F1-Score": f1}
    axes[0].axis("off")
    axes[0].set_title(f"Métricas - {nome}", fontsize=13, fontweight="bold")

    for i, (k, v) in enumerate(metricas.items()):
        y = 0.8 - i * 0.22
        axes[0].text(0.05, y, k, fontsize=12, va="center")
        axes[0].text(0.95, y, f"{v:.2%}", fontsize=14, fontweight="bold",
                      va="center", ha="right", color="#2b6cb0")
        axes[0].axhline(y - 0.08, color="#e2e8f0", linewidth=1)

    # --- Painel direito: matriz de confusão (heatmap) ---
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
                xticklabels=["Não Diabético", "Diabético"],
                yticklabels=["Não Diabético", "Diabético"],
                cbar=False, annot_kws={"fontsize": 12})
    axes[1].set_title(f"Matriz de Confusão - {nome}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Predito")
    axes[1].set_ylabel("Real")

    plt.tight_layout()
    plt.show()

    # Relatório completo de classificação (precisão, recall, f1 por classe)
    print(f"Relatório de Classificação - {nome}\n")
    print(classification_report(y_true, y_pred))

In [ ]:
avaliar_modelo("KNN", y_val, y_val_pred_knn)

In [ ]:
avaliar_modelo("SVM", y_val, y_val_pred_svm)

In [ ]:
avaliar_modelo("DT", y_val, y_val_pred_dt)

In [ ]:
avaliar_modelo("MLP", y_val, y_val_pred_mlp)

In [ ]:
"""> **KNN — Análise:** O KNN com threshold ajustado para 0,65 (em vez do padrão 0,5) foi uma escolha deliberada para reduzir falsos positivos — exigindo maior confiança do modelo antes de classificar alguém como diabético. Isso eleva a precisão mas reduz o recall, ou seja, o modelo erra menos ao acusar diabetes, mas deixa passar mais casos reais. Na matriz de confusão, é esperado observar um número menor de falsos positivos e um número maior de falsos negativos em comparação com os demais modelos. Para um problema de triagem de saúde, essa troca é discutível — um falso negativo (dizer que o paciente não tem diabetes quando tem) pode ser mais prejudicial que um falso positivo.

> **SVM — Análise:** O SVM utilizou o `LinearSVC` com calibração de probabilidades via `CalibratedClassifierCV`, SMOTE com `sampling_strategy=0.5` (menos agressivo que os demais) e `class_weight='balanced'`, combinando duas estratégias de combate ao desbalanceamento. O parâmetro `C=1.0` controla a regularização — valores maiores tornam o modelo mais sensível a cada ponto, valores menores generalizam mais. A matriz de confusão deve refletir um modelo mais equilibrado entre precisão e recall para a classe positiva.

> **Árvore de Decisão — Análise:** Os hiperparâmetros `max_depth=10` e `min_samples_leaf=20` foram definidos para controlar o overfitting — sem limitação de profundidade, a árvore memorizaria os dados de treino mas generalizaria mal. Com `max_depth=10`, o modelo captura relações relevantes sem se tornar excessivamente específico. O `min_samples_leaf=20` garante que cada folha da árvore tenha pelo menos 20 amostras, evitando decisões baseadas em casos isolados. Na matriz de confusão, é comum observar precisão menor (mais falsos positivos) dado que a árvore tende a ser mais agressiva na classe minoritária após o SMOTE.

> **MLP — Análise:** A rede neural foi configurada com duas camadas ocultas (64 e 32 neurônios), função de ativação ReLU e otimizador Adam, que adapta a taxa de aprendizado automaticamente. O `early_stopping=True` interrompe o treinamento quando a performance na validação interna para de melhorar, evitando overfitting e reduzindo o tempo de execução. Com `max_iter=300`, o modelo tem iterações suficientes para convergir. O MLP tende a capturar relações não-lineares mais complexas entre as variáveis, o que pode se refletir em um F1-Score mais equilibrado.

> **Comparativo Geral:**
"""

In [ ]:
# Tabela comparativa dos 4 modelos — métricas na validação
import pandas as pd

In [ ]:
linhas = []
for item in models_result:
    nome = item['model']
    y_pred = item['y_pred']
    linhas.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_val, y_pred),
        'Precisão': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1-Score': f1_score(y_val, y_pred)
    })

In [ ]:
df_comparativo = pd.DataFrame(linhas).set_index('Modelo')
df_comparativo = df_comparativo.style.format('{:.2%}').highlight_max(axis=0, color='#c6efce')
df_comparativo

In [ ]:
# Gráfico de barras comparativo entre os modelos
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
modelos = [r['model'] for r in models_result]
metricas_plot = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
fns = [accuracy_score, precision_score, recall_score, f1_score]

In [ ]:
valores = {fn.__name__: [fn(y_val, r['y_pred']) for r in models_result] for fn in fns}

In [ ]:
x = np.arange(len(modelos))
width = 0.2

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for i, (label, vals) in enumerate(zip(metricas_plot, valores.values())):
    ax.bar(x + i * width, vals, width, label=label)

In [ ]:
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(modelos)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Comparativo de Métricas entre os Modelos (Validação)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.tick_params(length=0)
ax.set_frame_on(False)
plt.tight_layout()
plt.show()

In [ ]:
"""## Avaliação Final no Conjunto de Teste

Após a seleção do melhor modelo com base nos resultados da validação, realizamos a avaliação final utilizando o conjunto de **teste** — dados que não foram vistos em nenhuma etapa anterior (nem no treino, nem na validação). Isso garante uma estimativa honesta da capacidade de generalização do modelo para dados reais e inéditos.
"""

Avaliação final do melhor modelo no conjunto de TESTE
O conjunto de teste só é usado aqui, ao final — nunca durante o desenvolvimento

In [ ]:
# Com base nos resultados da validação, o SVM foi identificado como o melhor modelo
# Realizamos aqui a predição sobre o conjunto de teste (dados nunca vistos antes)
y_test_pred_svm = svm_pipe.predict(X_test)

In [ ]:
avaliar_modelo("SVM — Conjunto de Teste", y_test, y_test_pred_svm)

In [ ]:
"""> **Avaliação no Teste — Análise:** Os resultados no conjunto de teste confirmam (ou não) a capacidade de generalização do SVM. Se as métricas forem próximas às obtidas na validação, o modelo está bem calibrado e generalizando adequadamente. Uma queda significativa nas métricas indicaria overfitting ao conjunto de validação, o que poderia ser corrigido com técnicas como cross-validation para seleção de hiperparâmetros.

---

# Discussão, análise crítica e conclusão


* Apresentar e discutir o modelo de melhor desempenho e justificar tecnicamente

* Apresentar e discutir as limitações e possíveis desafios.

# **Melhor modelo**
Dentre os modelos avaliados, o Support Vector Machine (SVM) apresentou o melhor desempenho geral, destacando-se pelo equilíbrio entre as métricas de precisão, recall e F1-score para a classe minoritária. Enquanto outros modelos demonstraram maior sensibilidade (recall) ou maior precisão isoladamente, o SVM foi capaz de manter um compromisso mais estável entre identificar corretamente os casos positivos e evitar classificações incorretas. Esse comportamento pode ser justificado tecnicamente pela capacidade do SVM de construir fronteiras de decisão mais robustas, especialmente em cenários com sobreposição entre classes, utilizando maximização de margem e, quando aplicável, transformações não lineares por meio de kernels. Dessa forma, o modelo se mostrou mais adequado ao problema, no qual há uma distribuição desbalanceada e padrões complexos nos dados.

# **Limitações e desafios**
Entretanto, apesar do desempenho superior, o modelo SVM ainda apresenta limitações importantes. Primeiramente, observa-se que, mesmo com técnicas de balanceamento como o uso do SMOTE, há uma dificuldade intrínseca na separação das classes, possivelmente devido à sobreposição entre indivíduos com e sem diabetes em determinadas regiões do espaço de atributos. E ainda, o SVM é sensível à escolha de hiperparâmetros, como o parâmetro de regularização e o tipo de kernel, o que pode impactar significativamente seu desempenho. Outro desafio relevante está relacionado à calibragem das probabilidades e ao ajuste do limiar de decisão, uma vez que pequenas variações podem afetar o equilíbrio entre precisão e recall. Por fim, destaca-se que modelos mais robustos, como métodos baseados em ensembles, podem ser explorados como alternativa, visto que tendem a capturar relações mais complexas e reduzir variância, especialmente em problemas com dados desbalanceados.
"""

In [ ]:
"""---

#Submissão

O notebook desenvolvido no Google Colab deve ser exportado em formato PDF por meio da opção:
Arquivo → Imprimir → Salvar como PDF.

É obrigatório que o arquivo PDF contenha todas as células processadas, exibindo os resultados, gráficos e métricas gerados durante a execução. Trabalhos submetidos com células não executadas ou sem resultados visíveis não serão avaliados.

A submissão deve seguir as seguintes orientações:

O envio deve ser feito por apenas um integrante do grupo, previamente cadastrado no AVA.

O notebook Colab original deve estar compartilhado publicamente (ou com permissão de visualização) e o link de acesso deve ser inserido abaixo:

🔗 LINK PARA O COLAB:
(https://colab.research.google.com/drive/1dumkx3Klosb7dTJoxeYF_HKUsbxi4yDZ?usp=sharing)

Essa etapa permite ao professor a verificação do código-fonte completo, dos procedimentos de análise e da reprodutibilidade dos resultados.

BOM TRABALHO!! ✅ !!!

---
"""